In [3]:
import numpy as np
import tensorflow as tf

# Load model (no compile needed)
model = tf.keras.models.load_model("model_fpga_ready.h5", compile=False)

# Extract weights again
weights = []
biases = []

for layer in model.layers:
    if "dense" in layer.name or "output" in layer.name:
        w, b = layer.get_weights()
        weights.append(w.astype(np.float32))
        biases.append(b.astype(np.float32))

print("Weights loaded successfully")

Weights loaded successfully


In [22]:
#SCALE = 2**8  # for Q8.8
SCALE = 2**10  # Q6.10

def quantize(x):
    return np.round(x * SCALE) / SCALE

In [23]:
weights_q = [quantize(w) for w in weights]
biases_q = [quantize(b) for b in biases]

In [24]:
def mlp_forward_fixed(x, weights, biases):
    x = quantize(x)

    z1 = quantize(np.dot(x, weights[0]) + biases[0])
    a1 = quantize(np.maximum(0, z1))

    z2 = quantize(np.dot(a1, weights[1]) + biases[1])
    a2 = quantize(np.maximum(0, z2))

    z3 = quantize(np.dot(a2, weights[2]) + biases[2])

    return z3

In [25]:
X = np.load("data/X_windows.npy")
y = np.load("data/y_targets.npy")

In [26]:
split_index = int(0.8 * len(X))

X_test = X[split_index:]
y_test = y[split_index:]

print("Test shape:", X_test.shape)

Test shape: (4065, 54)


In [27]:
def relu(x):
    return np.maximum(0, x)

In [28]:
def mlp_forward(x, weights, biases):
    z1 = np.dot(x, weights[0]) + biases[0]
    a1 = relu(z1)

    z2 = np.dot(a1, weights[1]) + biases[1]
    a2 = relu(z2)

    z3 = np.dot(a2, weights[2]) + biases[2]

    return z3

In [29]:
def mlp_forward_fixed(x, weights, biases):
    x = quantize(x)

    z1 = quantize(np.dot(x, weights[0]) + biases[0])
    a1 = quantize(relu(z1))

    z2 = quantize(np.dot(a1, weights[1]) + biases[1])
    a2 = quantize(relu(z2))

    z3 = quantize(np.dot(a2, weights[2]) + biases[2])

    return z3

In [30]:
num_samples = 5

for i in range(num_samples):
    x = X_test[i:i+1]

    float_pred = mlp_forward(x, weights, biases)
    fixed_pred = mlp_forward_fixed(x, weights_q, biases_q)

    print(f"\nSample {i}")
    print("Float :", float_pred[0][0])
    print("Fixed :", fixed_pred[0][0])
    print("Error :", abs(float_pred[0][0] - fixed_pred[0][0]))


Sample 0
Float : 46.01036
Fixed : 46.0459
Error : 0.03553772

Sample 1
Float : 43.743195
Fixed : 43.77832
Error : 0.035125732

Sample 2
Float : 42.07762
Fixed : 42.120117
Error : 0.042495728

Sample 3
Float : 49.533104
Fixed : 49.572266
Error : 0.039161682

Sample 4
Float : 49.985977
Fixed : 50.024414
Error : 0.03843689


In [31]:
SCALE = 2**8  # Q8.8

weights_int = [(w * SCALE).round().astype(np.int16) for w in weights]
biases_int = [(b * SCALE).round().astype(np.int16) for b in biases]

In [32]:
np.savetxt("data/W1.csv", weights_int[0], fmt="%d", delimiter=",")
np.savetxt("data/b1.csv", biases_int[0], fmt="%d", delimiter=",")

np.savetxt("data/W2.csv", weights_int[1], fmt="%d", delimiter=",")
np.savetxt("data/b2.csv", biases_int[1], fmt="%d", delimiter=",")

np.savetxt("data/W3.csv", weights_int[2], fmt="%d", delimiter=",")
np.savetxt("data/b3.csv", biases_int[2], fmt="%d", delimiter=",")

In [33]:
print("W1 range:", weights_int[0].min(), "to", weights_int[0].max())
print("W2 range:", weights_int[1].min(), "to", weights_int[1].max())
print("W3 range:", weights_int[2].min(), "to", weights_int[2].max())

W1 range: -207 to 178
W2 range: -142 to 157
W3 range: -172 to 115
